In [2]:
from pykeen.models.inductive import InductiveNodePieceGNN
from pykeen.training import SLCWATrainingLoop
from pykeen.evaluation.rank_based_evaluator import SampledRankBasedEvaluator
from pykeen.stoppers import EarlyStopper
from pykeen.losses import NSSALoss
from pykeen.triples import TriplesFactory

from torch.optim import Adam


/home/jwackito/miniconda3/envs/csv2pront/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
transductive_training_factory = TriplesFactory.from_path('dataset/ind_transductive_train.tsv',create_inverse_triples=True)
inductive_validation_factory = TriplesFactory.from_path('dataset/ind_inductive_validation.tsv',create_inverse_triples=True)
inductive_inference_factory = TriplesFactory.from_path('dataset/ind_inductive_validation.tsv',create_inverse_triples=True)

In [4]:
model = InductiveNodePieceGNN(
    triples_factory=transductive_training_factory,  # training factory, will be also used for a GNN
    validation_factory=inductive_validation_factory,
    inference_factory=inductive_inference_factory,  # inference factory, will be used for a GNN
    num_tokens=12,  # length of a node hash - how many unique relations per node will be used
    aggregation="mlp",  # aggregation function, defaults to an MLP, can be any PyTorch function
    loss=NSSALoss(margin=15),  # dummy loss
    random_seed=42,
    gnn_encoder=None,  # defaults to a 2-layer CompGCN with DistMult composition function
)
#model.to('cuda:1')

No symbolic computation of output shape.                     
No symbolic computation of output shape.              


In [5]:
optimizer = Adam(params=model.parameters(), lr=0.0005)

In [6]:
training_loop = SLCWATrainingLoop(
    triples_factory=transductive_training_factory,  # training triples
    model=model,
    optimizer=optimizer,
    negative_sampler_kwargs=dict(num_negs_per_pos=32),
    mode="training",   # necessary to specify for the inductive mode - training has its own set of nodes
)

In [7]:
# Validation and Test evaluators use a restricted protocol ranking against 50 random negatives
valid_evaluator = SampledRankBasedEvaluator(
    mode="validation",   # necessary to specify for the inductive mode - this will use inference nodes
    evaluation_factory=inductive_validation_factory,  # validation triples to predict
    additional_filter_triples=inductive_inference_factory.mapped_triples,   # filter out true inference triples
)

In [8]:
# According to the original code
# https://github.com/kkteru/grail/blob/2a3dffa719518e7e6250e355a2fb37cd932de91e/test_ranking.py#L526-L529
# test filtering uses only the inductive_inference split and does not include inductive_validation triples
# If you use the full RankBasedEvaluator, both inductive_inference and inductive_validation triples
# must be added to the additional_filter_triples
test_evaluator = SampledRankBasedEvaluator(
    mode="testing",   # necessary to specify for the inductive mode - this will use inference nodes
    evaluation_factory=inductive_validation_factory,  # test triples to predict
    additional_filter_triples=inductive_inference_factory.mapped_triples,   # filter out true inference triples
)

In [9]:
early_stopper = EarlyStopper(
    model=model,
    training_triples_factory=transductive_training_factory,
    evaluation_triples_factory=inductive_validation_factory,
    frequency=1,
    #patience=100000,  # for test reasons, turn it off
    result_tracker=None,
    evaluation_batch_size=256,
    evaluator=valid_evaluator,
)

In [10]:
# Training starts here
training_loop.train(
    triples_factory=transductive_training_factory,
    stopper=early_stopper,
    num_epochs=100,
    batch_size=256,
)

Training epochs on cpu:   0%|          | 0/100 [24:37:32<?, ?epoch/s]                  


KeyboardInterrupt: 

In [ ]:
# Test evaluation
result = test_evaluator.evaluate(
    model=model,
    mapped_triples=dataset.inductive_testing.mapped_triples,
    additional_filter_triples=dataset.inductive_inference.mapped_triples,
    batch_size=256,
)

In [ ]:
# print final results
print(result.to_flat_dict())

In [ ]:
dataset.transductive_training, dataset.inductive_inference

In [ ]:
dataset.inductive_testing.mapped_triples